In [0]:
date_path=dbutils.widgets.get("date_path")
oblist_path=dbutils.widgets.get("oblist_path")
office_path = dbutils.widgets.get("office_path")
client_path=dbutils.widgets.get("client_path")
cubeserviceofficetxnsourcesystem_path=dbutils.widgets.get("cubeserviceofficetxnsourcesystem_path")
payerdimension_path=dbutils.widgets.get("payerdimension_path")
mart_fact_accountreceivablebilled_path=dbutils.widgets.get("mart_fact_accountreceivablebilled_path")
alphacollector_claims_path=dbutils.widgets.get("alphacollector_claims_path")
stg_fact_accountreceivablebilled_path = dbutils.widgets.get("stg_fact_accountreceivablebilled_path")
mart_accountreceivablebilled = dbutils.widgets.get("mart_accountreceivablebilled")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {mart_fact_accountreceivablebilled_path}").collect()[0]['cnt']

if count == 0:
    print("Full load for fact_accountsreceivable_billed executed")

    spark.sql(f"""
    INSERT INTO {mart_fact_accountreceivablebilled_path}
    (
        reporting_date,
        source_system_key,
        office_key,
        payor_key,
        client_key,
        ar_balance,
        episode_start,
        episode_end,
        first_dos,
        last_dos,
        period_start,
        period_end,
        days_0_30,
        days_31_60,
        days_61_90,
        days_91_180,
        days_181_270,
        days_270_plus,
        reserve_required,
        age_from_today,
        age_from_quarter_end,
        invoice_number,
        invoice_date,
        last_billing_note,
        last_billing_note_type,
        reporting_date_key
    )
    SELECT
        TRY_CAST(ReportingDate AS DATE) AS reporting_date,
        TRY_CAST(SourceSystemKey AS INT) AS source_system_key,
        TRY_CAST(OfficeKey AS INT) AS office_key,
        TRY_CAST(PayorKey AS INT) AS payor_key,
        TRY_CAST(ClientKey AS INT) AS client_key,
        TRY_CAST(AR_Balance AS DOUBLE) AS ar_balance,

        TRY_CAST(EpisodeStart AS TIMESTAMP) AS episode_start,
        TRY_CAST(EpisodeEnd AS TIMESTAMP) AS episode_end,

        TRY_CAST(FirstDOS AS STRING) AS first_dos,
        TRY_CAST(LastDOS AS STRING) AS last_dos,

        TRY_CAST(PeriodStart AS TIMESTAMP) AS period_start,
        TRY_CAST(PeriodEnd AS TIMESTAMP) AS period_end,

        TRY_CAST(`0___30_Days` AS DOUBLE) AS days_0_30,
        TRY_CAST(`31___60_Days` AS DOUBLE) AS days_31_60,
        TRY_CAST(`61___90_Days` AS DOUBLE) AS days_61_90,
        TRY_CAST(`91___180_Days` AS DOUBLE) AS days_91_180,
        TRY_CAST(`181___270_Days` AS DOUBLE) AS days_181_270,
        TRY_CAST(`271__Days` AS DOUBLE) AS days_270_plus,

        TRY_CAST(ReserveRequired AS DOUBLE) AS reserve_required,
        TRY_CAST(AgeFromToday AS INT) AS age_from_today,
        TRY_CAST(AgeFromQuarterEnd AS INT) AS age_from_quarter_end,
        TRY_CAST(InvNum AS STRING) AS invoice_number,
        TRY_CAST(InvDate AS TIMESTAMP) AS invoice_date,
        TRY_CAST(LastBillingNote AS STRING) AS last_billing_note,
        TRY_CAST(LastBillingNoteType AS STRING) AS last_billing_note_type,
        TRY_CAST(ReportingDateKey AS INT) AS reporting_date_key

    FROM {mart_accountreceivablebilled}
    WHERE SourceSystemKey IN (0, 19, 6);
    """)


In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 5:
  
  spark.sql(
      f""" 
  CREATE OR REPLACE TEMPORARY VIEW last_qtr AS
  SELECT
    MAX(WeekEndingDate) AS LastDayInQtr
  FROM {date_path}
  WHERE FiscalQuarterNbr = (
    SELECT FiscalQuarterNbr
    FROM {date_path}
    WHERE CalendarDate = date_add(current_date(), -4)
  );
  """
  )

  spark.sql(
      f"""
  CREATE OR REPLACE TEMPORARY VIEW tmp AS
  SELECT
    ReportingWeekendingDate,
    s.SourceSystemKey,
    ofc.OfficeKey,
    p.PayerKey,
    c.ClientKey,
    TotalDue AS AR_Balance,

    CASE WHEN InvDate BETWEEN date_add(current_date(), -30) AND current_date()
        THEN coalesce(TotalDue,0) END AS 0_30_Days,

    CASE WHEN InvDate BETWEEN date_add(current_date(), -60) AND date_add(current_date(), -31)
        THEN coalesce(TotalDue,0) END AS 31_60_Days,

    CASE WHEN InvDate BETWEEN date_add(current_date(), -90) AND date_add(current_date(), -61)
        THEN coalesce(TotalDue,0) END AS 61_90_Days,

    CASE WHEN InvDate BETWEEN date_add(l.LastDayInQtr,-181) AND date_add(current_date(), -91)
        THEN coalesce(TotalDue,0) END AS 91_180_Days,

    CASE WHEN InvDate BETWEEN date_add(l.LastDayInQtr,-271) AND date_add(l.LastDayInQtr,-182)
        THEN coalesce(TotalDue,0) END AS 181_270_days,

    CASE WHEN InvDate <= date_add(l.LastDayInQtr,-272)
        THEN TotalDue END AS 270plus_Days,

    datediff(current_date(), InvDate) AS AgefromToday,
    datediff(l.LastDayInQtr, InvDate) AS AgefromQuarterEnd,
    InvNo,
    InvDate
  FROM {oblist_path} ob
  LEFT JOIN {office_path} ofc 
    ON ofc.OfficeNumber = ob.Office
  LEFT JOIN (
    SELECT * 
    FROM {client_path} 
    WHERE ClientKey <> 2736498
  ) c 
    ON c.SourceSystemId = ob.ClientNo 
    AND c.OfficeNumber = ob.Office  -- ADD THIS CONDITION
  LEFT JOIN {cubeserviceofficetxnsourcesystem_path} s 
    ON s.SourceSystemName = 'BEARS'
  LEFT JOIN {payerdimension_path} p 
    ON p.PayerID = ob.BillTo
    CROSS JOIN last_qtr l;
  """
  )

  spark.sql(
  f"""
  INSERT INTO {mart_fact_accountreceivablebilled_path}
  (
    reporting_date,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    ar_balance,
    episode_start,
    episode_end,
    first_dos,
    last_dos,
    period_start,
    period_end,
    days_0_30,
    days_31_60,
    days_61_90,
    days_91_180,
    days_181_270,
    days_270_plus,
    reserve_required,
    age_from_today,
    age_from_quarter_end,
    invoice_number,
    invoice_date,
    last_billing_note,
    last_billing_note_type,
    reporting_date_key
  )
  SELECT
    TRY_CAST(ReportingWeekendingDate AS DATE) AS reporting_date,
    TRY_CAST(SourceSystemKey AS INT) AS source_system_key,
    TRY_CAST(OfficeKey AS INT) AS office_key,
    TRY_CAST(PayerKey AS INT) AS payor_key,
    TRY_CAST(ClientKey AS INT) AS client_key,
    TRY_CAST(AR_Balance AS DOUBLE) AS ar_balance,

    TRY_CAST(NULL AS TIMESTAMP) AS episode_start,
    TRY_CAST(NULL AS TIMESTAMP) AS episode_end,
    TRY_CAST(NULL AS STRING) AS first_dos,
    TRY_CAST(NULL AS STRING) AS last_dos,
    TRY_CAST(NULL AS TIMESTAMP) AS period_start,
    TRY_CAST(NULL AS TIMESTAMP) AS period_end,

    TRY_CAST(COALESCE(`0_30_Days`, 0) AS DOUBLE) AS days_0_30,
    TRY_CAST(COALESCE(`31_60_Days`, 0) AS DOUBLE) AS days_31_60,
    TRY_CAST(COALESCE(`61_90_Days`, 0) AS DOUBLE) AS days_61_90,
    TRY_CAST(COALESCE(`91_180_Days`, 0) AS DOUBLE) AS days_91_180,
    TRY_CAST(COALESCE(`181_270_days`, 0) AS DOUBLE) AS days_181_270,
    TRY_CAST(COALESCE(`270plus_Days`, 0) AS DOUBLE) AS days_270_plus,

    TRY_CAST(
        COALESCE(`181_270_days`, 0) + COALESCE(`270plus_Days`, 0)
        AS DOUBLE
    ) AS reserve_required,

    TRY_CAST(AgefromToday AS INT) AS age_from_today,
    TRY_CAST(AgefromQuarterEnd AS INT) AS age_from_quarter_end,

    TRY_CAST(invno AS STRING) AS invoice_number,
    TRY_CAST(InvDate AS TIMESTAMP) AS invoice_date,

    TRY_CAST(NULL AS STRING) AS last_billing_note,
    TRY_CAST(NULL AS STRING) AS last_billing_note_type,

    TRY_CAST(DATE_FORMAT(CAST(ReportingWeekendingDate AS DATE), 'yyyyMMdd') AS INT) AS reporting_date_key

  FROM tmp
  """
  )



  spark.sql(
      f"""
  CREATE OR REPLACE TEMPORARY VIEW ar AS
  SELECT
    ReportingWeekendingDate,
    SourceSystemKey,
    OfficeKey,
    PayerKey,
    ClientKey,
    AR_Balance,
    coalesce(0_30_Days,0) AS 0_30_Days,
    coalesce(31_60_Days,0) AS 31_60_Days,
    coalesce(61_90_Days,0) AS 61_90_Days,
    coalesce(91_180_Days,0) AS 91_180_Days,
    coalesce(181_270_days,0) AS 181_270_days,
    coalesce(270plus_Days,0) AS 270plus_Days,
    coalesce(181_270_days,0) + coalesce(270plus_Days,0) AS ReserveRequired,
    AgefromToday,
    AgefromQuarterEnd,
    InvNo,
    InvDate
  FROM tmp;
  """
  )


In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 5: 
  spark.sql( 
          f""" 
  INSERT INTO {mart_fact_accountreceivablebilled_path}
  (
    reporting_date,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    ar_balance,
    episode_start,
    episode_end,
    first_dos,
    last_dos,
    period_start,
    period_end,
    days_0_30,
    days_31_60,
    days_61_90,
    days_91_180,
    days_181_270,
    days_270_plus,
    reserve_required,
    age_from_today,
    age_from_quarter_end,
    invoice_number,
    invoice_date,
    last_billing_note,
    last_billing_note_type,
    reporting_date_key
  )
  SELECT
    TRY_CAST(ReportingDate AS DATE) AS reporting_date,
    TRY_CAST(SourceSystemKey AS INT) AS source_system_key,
    TRY_CAST(OfficeKey AS INT) AS office_key,
    TRY_CAST(PayerKey AS INT) AS payor_key,
    TRY_CAST(ClientKey AS INT) AS client_key,
    TRY_CAST(Balance AS DOUBLE) AS ar_balance,

    TRY_CAST(NULL AS TIMESTAMP) AS episode_start,
    TRY_CAST(NULL AS TIMESTAMP) AS episode_end,
    TRY_CAST(NULL AS STRING) AS first_dos,
    TRY_CAST(NULL AS STRING) AS last_dos,
    TRY_CAST(NULL AS TIMESTAMP) AS period_start,
    TRY_CAST(NULL AS TIMESTAMP) AS period_end,

    TRY_CAST(COALESCE(`0_30_Days`, 0) AS DOUBLE) AS days_0_30,
    TRY_CAST(COALESCE(`31_60_Days`, 0) AS DOUBLE) AS days_31_60,
    TRY_CAST(COALESCE(`61_90_Days`, 0) AS DOUBLE) AS days_61_90,
    TRY_CAST(COALESCE(`91_180_Days`, 0) AS DOUBLE) AS days_91_180,
    TRY_CAST(COALESCE(`181_270_days`, 0) AS DOUBLE) AS days_181_270,
    TRY_CAST(COALESCE(`270plus_Days`, 0) AS DOUBLE) AS days_270_plus,

    TRY_CAST(
        COALESCE(`181_270_days`, 0) + COALESCE(`270plus_Days`, 0)
        AS DOUBLE
    ) AS reserve_required,

    TRY_CAST(AgefromToday AS INT) AS age_from_today,
    TRY_CAST(AgefromQuarterEnd AS INT) AS age_from_quarter_end,

    TRY_CAST(ClaimNumber AS STRING) AS invoice_number,
    TRY_CAST(InvDate AS TIMESTAMP) AS invoice_date,

    TRY_CAST(NULL AS STRING) AS last_billing_note,
    TRY_CAST(NULL AS STRING) AS last_billing_note_type,

    TRY_CAST(DATE_FORMAT(CAST(ReportingDate AS DATE), 'yyyyMMdd') AS INT) AS reporting_date_key
    
  FROM (
    SELECT
      DATE_ADD(CURRENT_DATE(), -4) AS ReportingDate,
      19 AS SourceSystemKey,
      ofc.OfficeKey,
      pd.PayerKey,
      clt.ClientKey,
      Balance,

      CASE WHEN DateBilled_converted BETWEEN DATE_ADD(CURRENT_DATE(), -30) AND CURRENT_DATE()
          THEN Balance END AS 0_30_Days,

      CASE WHEN DateBilled_converted BETWEEN DATE_ADD(CURRENT_DATE(), -60) AND DATE_ADD(CURRENT_DATE(), -31)
          THEN Balance END AS 31_60_Days,

      CASE WHEN DateBilled_converted BETWEEN DATE_ADD(CURRENT_DATE(), -90) AND DATE_ADD(CURRENT_DATE(), -61)
          THEN Balance END AS 61_90_Days,

      CASE WHEN DateBilled_converted BETWEEN DATE_ADD(l.LastDayInQtr, -181) AND DATE_ADD(CURRENT_DATE(), -91)
          THEN Balance END AS 91_180_Days,

      CASE WHEN DateBilled_converted BETWEEN DATE_ADD(l.LastDayInQtr, -271) AND DATE_ADD(l.LastDayInQtr, -182)
          THEN Balance END AS 181_270_days,

      CASE WHEN DateBilled_converted <= DATE_ADD(l.LastDayInQtr, -272)
          THEN Balance END AS 270plus_Days,

      DATEDIFF(CURRENT_DATE(), DateBilled_converted) AS AgefromToday,
      DATEDIFF(l.LastDayInQtr, DateBilled_converted) AS AgefromQuarterEnd,
      ClaimNumber,
      DateBilled_converted AS InvDate
    FROM (
      SELECT 
        *,
        TO_DATE(DateBilled, 'MM/dd/yyyy') AS DateBilled_converted
      FROM {alphacollector_claims_path}
    ) clm
    LEFT JOIN {office_path} ofc
      ON clm.OfficeExternalId = ofc.OfficeNumber
    LEFT JOIN (
      SELECT Name, PayerKey
      FROM (
        SELECT *,
              ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey DESC) rnk
        FROM {payerdimension_path}
        WHERE SourceSystemKey = 19
      ) WHERE rnk = 1
    ) pd ON pd.Name = clm.PayerName
    LEFT JOIN (
      SELECT *
      FROM (
        SELECT ClientKey, MedicalRecordNumber,
              ROW_NUMBER() OVER (PARTITION BY MedicalRecordNumber ORDER BY ClientKey DESC) rnk
        FROM {client_path}
        WHERE SourceSystem = 'CUBHUB'
      ) a WHERE rnk = 1
    ) clt ON clt.MedicalRecordNumber = clm.MedicalRecordNumber
    CROSS JOIN last_qtr l
  ) a;
  """
  )


In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 5:  
  spark.sql(
      f"""
  UPDATE {mart_fact_accountreceivablebilled_path}
  SET reporting_date_key = replace(cast(reporting_date as string), '-', '')
  WHERE source_system_key IN (0, 19)
    AND reporting_date_key IS NULL;
      """
  )

In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 5: 

  spark.sql(f"""
  INSERT INTO {mart_fact_accountreceivablebilled_path} (
    reporting_date, 
    source_system_key, 
    office_key, 
    payor_key, 
    client_key, 
    ar_balance,
    episode_start, 
    episode_end, 
    first_dos, 
    last_dos, 
    period_start, 
    period_end, 
    days_0_30, 
    days_31_60, 
    days_61_90, 
    days_91_180, 
    days_181_270, 
    days_270_plus, 
    reserve_required,
    age_from_today, 
    age_from_quarter_end, 
    invoice_number, 
    invoice_date, 
    last_billing_note, 
    last_billing_note_type,
    reporting_date_key
  )
  SELECT
    CAST(reportingdate AS DATE) AS reporting_date, 
    CAST(sourcesystemkey AS INT) AS source_system_key, 
    CAST(officekey AS INT) AS office_key, 
    CAST(payorkey AS INT) AS payor_key, 
    CAST(clientkey AS INT) AS client_key, 
    CAST(ar_balance AS DOUBLE) AS ar_balance,

    CAST(episodestart AS TIMESTAMP) AS episode_start, 
    CAST(episodeend AS TIMESTAMP) AS episode_end, 
    CAST(firstdos AS STRING) AS first_dos, 
    CAST(lastdos AS STRING) AS last_dos, 
    CAST(periodstart AS TIMESTAMP) AS period_start, 
    CAST(periodend AS TIMESTAMP) AS period_end, 

    CAST(days_0_30 AS DOUBLE) AS days_0_30, 
    CAST(days_31_60 AS DOUBLE) AS days_31_60, 
    CAST(days_61_90 AS DOUBLE) AS days_61_90, 
    CAST(days_91_180 AS DOUBLE) AS days_91_180, 
    CAST(days_181_270 AS DOUBLE) AS days_181_270, 
    CAST(days_270_plus AS DOUBLE) AS days_270_plus,

    CAST(reserverequired AS DOUBLE) AS reserve_required,
    CAST(agefromtoday AS INT) AS age_from_today, 
    CAST(agefromquarterend AS INT) AS age_from_quarter_end, 

    CAST(invnum AS STRING) AS invoice_number, 
    CAST(invdate AS TIMESTAMP) AS invoice_date, 
    CAST(lastbillingnote AS STRING) AS last_billing_note, 
    CAST(lastbillingnotetype AS STRING) AS last_billing_note_type,

    CAST(reportingdatekey AS INT) AS reporting_date_key
  FROM {stg_fact_accountreceivablebilled_path}
  WHERE reportingdate = (
        SELECT MAX(reportingdate)
        FROM {stg_fact_accountreceivablebilled_path}
    )
  """)
